# Benchmark mfego sur Hartmann 6D — comparaison avec scikit-learn, SMT et BoTorch

Ce notebook compare **mfego** (co-krigeage récursif non imbriqué de Sacher et al. 2021, ρ calculé à chaque niveau) à trois
bibliothèques de processus gaussiens existantes, sur le problème multi-fidélité de Hartmann 6D de l'article (Éq. 30-32) :

| Bibliothèque | Modèles utilisés | Rôle dans la comparaison |
|---|---|---|
| **scikit-learn** | `GaussianProcessRegressor` (RBF ARD + bruit blanc) | GP mono-fidélité de référence (HF seule) |
| **SMT** | `KRG` (HF seule), `MFK` (co-krigeage récursif de Le Gratiet, ρ par GLS) | même théorie que mfego → comparaison directe des surrogates et des ρ̂ |
| **BoTorch** | `SingleTaskGP`, `SingleTaskMultiFidelityGP`, `qLogExpectedImprovement`, `qMultiFidelityKnowledgeGradient` | référence de l'optimisation bayésienne (mono- et multi-fidélité, coûts pris en compte) |

Trois parties : **(1)** précision des surrogates en fonction du nombre de points HF, **(2)** optimisation à **budget de coût égal**,
**(3)** vitesse de mfego avant/après la revue.

**Environnement** : venv dédié (voir `requirements-benchmark.txt`) :
```
python -m venv .venv-benchmark
.venv-benchmark\Scripts\python -m pip install -r requirements-benchmark.txt
```
puis choisir l'interpréteur / noyau `.venv-benchmark`. Le drapeau `QUICK` ci-dessous réduit graines et budgets
(quelques minutes) ; `QUICK = False` reproduit les résultats enregistrés (calcul parallèle, ~1 h dominée par BoTorch MF-KG).

In [1]:
import json
import sys
import time
import warnings
from importlib.metadata import version
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore")
pio.renderers.default = "plotly_mimetype+notebook_connected"
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "mfego" / "src").is_dir())
sys.path.insert(0, str(ROOT / "benchmarks"))
import bench_lib as bl  # noqa: E402

QUICK = False                        # True: quick run (1 seed, small budget)
SEEDS = [0] if QUICK else [0, 1, 2, 3, 4]
N_HF_LIST = [10, 20] if QUICK else [10, 20, 40]
BUDGET = 180 if QUICK else 250       # total cost of an optimization run (DOE included)
PROBLEM = {"ks": (1, np.inf), "delta": 0.05, "costs": (1.0, 10.0)}
MF_DOE, SF_DOE = [20, 10], [12]      # same initial cost: 20 x 1 + 10 x 10 = 12 x 10 = 120
FIG_DIR = ROOT / "benchmarks" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

pd.DataFrame({p: [version(p)] for p in ["numpy", "scipy", "scikit-learn", "smt", "torch", "botorch",
                                        "gpytorch", "plotly"]}, index=["version"])

,numpy,scipy,scikit-learn,smt,torch,botorch,gpytorch,plotly
version,2.5.3,1.18.1,1.9.1,2.15.0,2.14.1,0.18.1,1.15.2,7.1.0


## 0. Problème

Hartmann 6D sur $[0,1]^6$ (Éq. 30, minimum $f^* = -3{,}32237$ en $x^* = (0{,}2017;\,0{,}1500;\,0{,}4769;\,0{,}2753;\,0{,}3117;\,0{,}6573)$).
Les niveaux de fidélité sont ceux de Sacher (Éq. 31-32) : $f^{(l)}(x) = U_{k_l}(x + \delta/k_l)$ avec la suite
$U_{k+1} = \tfrac12 (f^2/U_k + U_k)$, $U_0 = -5$. Ici **2 niveaux** : $k_1 = 1$ (basse fidélité, coût 1) et $k_2 = \infty$
(Hartmann, coût 10), décalage $\delta = 0{,}05$. Les plans d'expériences initiaux sont **non imbriqués** (un LHS par niveau).

In [2]:
problem = bl.HartmannMF(**PROBLEM)
x_check = np.random.default_rng(0).random((3000, 6))
print(f"f(x*) = {bl.hartmann6(bl.H6_XOPT)[0]:.6f}  (f* = {bl.H6_MIN:.6f})")
print(f"corrélation f(1) / f(2) sur 3000 points : {np.corrcoef(problem(x_check, 1), problem(x_check, 2))[0, 1]:.3f}")

f(x*) = -3.322368  (f* = -3.322368)
corrélation f(1) / f(2) sur 3000 points : 0.817


## 1. Précision des surrogates

Pour chaque taille $n_{HF} \in$ `N_HF_LIST` (avec $n_{BF} = 2\,n_{HF}$) et chaque graine, chaque modèle est ajusté sur le même
plan d'expériences puis évalué sur **2 000 points de test** (LHS) de la fonction HF :

* **RMSE / std(f)** (1 = pas mieux que la moyenne), **NLPD** (log-densité prédictive négative : précision *et* calibration de
  l'incertitude), **couverture à 95 %** (fraction des points de test dans l'intervalle prédit, idéal ≈ 0,95) ;
* temps d'ajustement et de prédiction ; pour les modèles multi-fidélité récursifs, le **ρ̂** estimé (mfego vs SMT MFK).

Les modèles mono-fidélité (SF) n'utilisent que les points HF.

In [3]:
start = time.perf_counter()
tasks = [(name, 2 * n_hf, n_hf, seed, PROBLEM) for name in bl.MODELS
         for n_hf in N_HF_LIST for seed in SEEDS]
accuracy = pd.DataFrame(bl.run_parallel(bl.accuracy_task, tasks))
print(f"{len(tasks)} ajustements en {time.perf_counter() - start:.0f} s")
if "error" in accuracy:
    display(accuracy[accuracy["error"].notna()][["model", "n_hf", "seed", "error"]])
summary = (accuracy.dropna(subset=["rmse_rel"])
           .groupby(["n_hf", "model"])[["rmse_rel", "nlpd", "coverage_95", "fit_time_s", "predict_time_s"]]
           .agg(["mean", "std"]).round(3))
summary

105 ajustements en 30 s


rmse_rel            nlpd          coverage_95  \
                                   mean    std     mean      std        mean   
n_hf model                                                                     
10   BoTorch MF-GP (MF)           0.977  0.044    0.483    0.061       0.952   
     BoTorch SingleTaskGP (SF)    0.971  0.056    0.751    0.207       0.884   
     SMT KRG (SF)                 1.098  0.128  330.095  697.981       0.609   
     SMT MFK (MF)                 1.185  0.149   43.449   72.911       0.586   
     mfego MF                     1.083  0.114    7.541    7.729       0.713   
     mfego SF                     1.057  0.095    6.674    7.518       0.743   
     scikit-learn GP (SF)         1.100  0.125   13.345    7.524       0.674   
20   BoTorch MF-GP (MF)           0.906  0.020    1.244    0.389       0.862   
     BoTorch SingleTaskGP (SF)    0.984  0.038    3.290    0.826       0.733   
     SMT KRG (SF)                 1.015  0.051   36.708   55.489       0.636   
     SMT MFK (MF)                 0.998  0.051    5.040    2.344       0.706   
     mfego MF                     0.994  0.051   10.959    6.585       0.630   
     mfego SF                     1.006  0.039   10.744   11.726       0.692   
     scikit-learn GP (SF)         1.080  0.032  188.704  333.770       0.637   
40   BoTorch MF-GP (MF)           0.833  0.047    1.174    1.230       0.881   
     BoTorch SingleTaskGP (SF)    0.904  0.054    1.151    1.012       0.861   
     SMT KRG (SF)                 0.928  0.107    1.894    1.936       0.831   
     SMT MFK (MF)                 1.037  0.185    1.798    1.469       0.807   
     mfego MF                     0.913  0.097    1.476    1.435       0.856   
     mfego SF                     1.022  0.022    1.578    1.465       0.831   
     scikit-learn GP (SF)         1.033  0.046    2.750    1.992       0.783   

                                      fit_time_s        predict_time_s         
                                  std       mean    std           mean    std  
n_hf model                                                                     
10   BoTorch MF-GP (MF)         0.014      3.410  0.282          1.128  0.036  
     BoTorch SingleTaskGP (SF)  0.050      5.452  0.543          0.133  0.008  
     SMT KRG (SF)               0.156      8.139  0.242          0.424  0.364  
     SMT MFK (MF)               0.214     11.834  0.415          0.027  0.008  
     mfego MF                   0.057      0.189  0.018          0.007  0.000  
     mfego SF                   0.202      0.065  0.023          0.003  0.001  
     scikit-learn GP (SF)       0.122      1.180  0.050          0.002  0.000  
20   BoTorch MF-GP (MF)         0.019      2.324  0.404          1.086  0.082  
     BoTorch SingleTaskGP (SF)  0.043      4.368  0.374          0.140  0.009  
     SMT KRG (SF)               0.109      9.548  0.618          0.321  0.098  
     SMT MFK (MF)               0.052     15.086  1.095          0.073  0.015  
     mfego MF                   0.078      0.270  0.034          0.019  0.003  
     mfego SF                   0.125      0.090  0.021          0.006  0.000  
     scikit-learn GP (SF)       0.193      1.121  0.050          0.003  0.000  
40   BoTorch MF-GP (MF)         0.057      3.610  0.852          0.395  0.246  
     BoTorch SingleTaskGP (SF)  0.050      3.185  0.192          0.134  0.026  
     SMT KRG (SF)               0.064     11.419  0.366          0.132  0.054  
     SMT MFK (MF)               0.051     18.334  0.622          0.035  0.002  
     mfego MF                   0.081      0.517  0.043          0.040  0.005  
     mfego SF                   0.068      0.192  0.020          0.011  0.001  
     scikit-learn GP (SF)       0.065      1.400  0.131          0.003  0.000

In [4]:
metrics = [("rmse_rel", "RMSE / std(f)"), ("nlpd", "NLPD (log)"), ("coverage_95", "couverture 95 %"),
           ("fit_time_s", "temps d'ajustement (s)")]
fig = make_subplots(rows=1, cols=4, subplot_titles=[t for _, t in metrics])
palette = dict(zip(bl.MODELS, ["#1f77b4", "#aec7e8", "#2ca02c", "#ff7f0e", "#ffbb78", "#9467bd", "#c5b0d5"]))
ok = accuracy.dropna(subset=["rmse_rel"])
for col, (key, _) in enumerate(metrics, start=1):
    for name in bl.MODELS:
        grp = ok[ok["model"] == name].groupby("n_hf")[key]
        values = grp.median() if key != "nlpd" else np.log10(grp.median().clip(lower=1e-3) + 1)
        fig.add_trace(go.Scatter(x=values.index, y=values.values, mode="lines+markers", name=name,
                                 legendgroup=name, showlegend=col == 1,
                                 line={"color": palette[name], "dash": "dash" if "SF" in name else "solid"}),
                      row=1, col=col)
fig.update_xaxes(title_text="n_HF")
fig.update_yaxes(type="log", row=1, col=4)
fig.update_layout(title="Précision des surrogates sur Hartmann 6D (médiane des graines ; pointillés = mono-fidélité)",
                  template="plotly_white", height=460)
fig.write_html(FIG_DIR / "accuracy.html", include_plotlyjs="cdn")
fig

In [5]:
rho = ok[ok["model"].isin(["mfego MF", "SMT MFK (MF)"])].copy()
rho["rho"] = rho["rhos"].apply(lambda r: r[0] if len(r) else np.nan)
rho.pivot_table(index=["n_hf", "seed"], columns="model", values="rho").round(3)

model      SMT MFK (MF)  mfego MF
n_hf seed                        
10   0           -0.821     0.999
     1           -4.679     0.998
     2            0.721     0.981
     3           -4.590     1.000
     4            4.216     1.043
20   0            1.049     0.989
     1           -0.211     0.909
     2            6.010     1.043
     3            1.786     0.931
     4            8.854     1.033
40   0           -0.754     0.866
     1            6.998     0.958
     2            1.769     0.965
     3            4.030     1.000
     4           -0.218     0.997

**Lecture — partie 1** (5 graines, 2 000 points de test ; les temps sont mesurés avec un seul thread par processus, car les
ajustements tournent en parallèle) :

* **Hartmann 6D est difficile avec peu de points** : avec 10 ou 20 points HF, aucun modèle ne fait nettement mieux que la
  moyenne (RMSE/std ≈ 0,9-1,2). Avec 40 points HF, les modèles multi-fidélité commencent à exploiter la basse fidélité :
  BoTorch MF-GP 0,83, mfego MF 0,91, contre 1,02 pour mfego SF et 1,03 pour scikit-learn.
* **BoTorch MF-GP est le plus précis et surtout le mieux calibré** : couverture à 95 % de 0,95 / 0,86 / 0,88 et NLPD de
  0,5 à 1,2, contre 0,71 / 0,63 / 0,86 et 7,5 / 11,0 / 1,5 pour mfego MF. Avec très peu de points, mfego (maximum de
  vraisemblance sans prior) est **trop confiant**. Les priors Gamma de BoTorch sur les longueurs de corrélation et sa borne
  basse du bruit (≥ 10⁻⁴) régularisent ce cas.
* **mfego MF fait mieux que SMT MFK**, qui repose sur la même théorie (récursivité de Le Gratiet) : RMSE 1,08 / 0,99 / 0,91
  contre 1,19 / 1,00 / 1,04. Surtout, **ρ̂ est stable avec mfego** (0,87 à 1,04, cohérent avec une corrélation BF/HF de 0,82),
  alors que **SMT MFK donne des ρ̂ erratiques** (de −4,7 à +8,9). mfego profile ρ par GLS à chaque évaluation de la vraisemblance
  et renormalise à ρ̂ ; SMT estime un modèle de tendance plus riche, mal identifié avec si peu de points HF.
* **mfego est de loin le plus rapide à ajuster** : 0,19 à 0,52 s (MF), contre 2,3 à 3,6 s pour BoTorch MF-GP, 1,1 à 1,4 s pour
  scikit-learn (SF) et 12 à 18 s pour SMT MFK.

## 2. Optimisation à budget de coût égal

Chaque méthode part d'un plan d'expériences de **même coût initial** (120) : 20 BF + 10 HF pour les méthodes multi-fidélité,
12 HF pour les méthodes mono-fidélité ; elle s'arrête quand le coût cumulé atteint `BUDGET`.

| Méthode | Modèle | Critère d'enrichissement |
|---|---|---|
| **mfego NN-MF-EGO** | co-krigeage récursif non imbriqué, ρ calculé | mérite de Sacher (Éq. 24 : AEI × W_L/W_l × réduction de variance) |
| **mfego SF-EGO** | GP mfego, HF seule | AEI (Éq. 20) |
| **BoTorch qLogEI (SF)** | `SingleTaskGP`, HF seule | log-Expected Improvement |
| **BoTorch MF-KG** | `SingleTaskMultiFidelityGP` | knowledge gradient multi-fidélité pondéré par le coût (modèle de coût affine = 1 / 10) |

Mesures : **erreur de la meilleure observation HF** $\min y_{HF} - f^*$ en fonction du coût, et **erreur de la recommandation
finale** $f(\hat x) - f^*$ où $\hat x$ minimise la moyenne prédite du niveau HF (même optimiseur pour toutes les méthodes).
Les runs sont exécutés en parallèle (un processus par run).

In [6]:
start = time.perf_counter()
methods = ["mfego NN-MF-EGO", "mfego SF-EGO", "BoTorch qLogEI (SF)", "BoTorch MF-KG"]
tasks = [(m, seed, BUDGET, PROBLEM, (MF_DOE, SF_DOE)) for m in methods for seed in SEEDS]
histories = bl.run_parallel(bl.optimization_task, tasks)
print(f"{len(tasks)} runs en {time.perf_counter() - start:.0f} s")
failed = [h for h in histories if "error" in h]
for h in failed:
    print("échec :", h["method"], h["seed"], h["error"])
histories = [h for h in histories if "error" not in h]
rows = []
for h in histories:
    levels = np.array(h["levels"][1:])
    rows.append({"method": h["method"], "seed": h["seed"], "final cost": h["cost"][-1],
                 "best observed HF error": h["best_observed"][-1],
                 "recommendation error": h["recommendation_error"], "iterations": len(levels),
                 "share of LF evaluations": float(np.mean(levels == 1)) if len(levels) else np.nan,
                 "mean time / iteration (s)": float(np.mean(h["iteration_time_s"][1:])) if len(levels) else np.nan})
results = pd.DataFrame(rows)
results.groupby("method").agg(["median", "mean"]).drop(columns=["seed"]).round(4)

20 runs en 2537 s


final cost        best observed HF error          \
                        median   mean                 median    mean   
method                                                                 
BoTorch MF-KG            250.0  251.8                 2.4474  2.3379   
BoTorch qLogEI (SF)      250.0  250.0                 1.2223  1.2167   
mfego NN-MF-EGO          257.0  256.2                 0.3259  0.9362   
mfego SF-EGO             250.0  250.0                 1.7385  1.7427   

                    recommendation error         iterations        \
                                  median    mean     median  mean   
method                                                              
BoTorch MF-KG                     1.2333  1.7057       75.0  74.2   
BoTorch qLogEI (SF)               1.4671  1.1869       13.0  13.0   
mfego NN-MF-EGO                   0.3319  0.9569       39.0  37.2   
mfego SF-EGO                      1.8453  1.9865       13.0  13.0   

                    share of LF evaluations         mean time / iteration (s)  \
                                     median    mean                    median   
method                                                                          
BoTorch MF-KG                        0.9067  0.8895                   18.5854   
BoTorch qLogEI (SF)                  0.0000  0.0000                    0.8792   
mfego NN-MF-EGO                      0.7179  0.5986                    0.4379   
mfego SF-EGO                         0.0000  0.0000                    0.1555   

                              
                        mean  
method                        
BoTorch MF-KG        19.4410  
BoTorch qLogEI (SF)   0.8495  
mfego NN-MF-EGO       0.3760  
mfego SF-EGO          0.1547

In [7]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Erreur de la meilleure observation HF (médiane, 25-75 %)",
                                                    "Erreur de la recommandation finale (par graine)"))
colors = dict(zip(methods, ["#1f77b4", "#aec7e8", "#9467bd", "#d62728"]))
grid = np.linspace(120, BUDGET, 60)
for method in methods:
    runs = [h for h in histories if h["method"] == method]
    if not runs:
        continue
    curves = np.array([np.interp(grid, h["cost"], np.maximum(h["best_observed"], 1e-6)) for h in runs])
    q25, q50, q75 = np.percentile(curves, [25, 50, 75], axis=0)
    fig.add_trace(go.Scatter(x=np.concatenate([grid, grid[::-1]]), y=np.concatenate([q75, q25[::-1]]), fill="toself",
                             line={"width": 0}, opacity=0.15, fillcolor=colors[method], showlegend=False,
                             hoverinfo="skip"), row=1, col=1)
    fig.add_trace(go.Scatter(x=grid, y=q50, name=method, line={"color": colors[method]}), row=1, col=1)
    fig.add_trace(go.Box(y=[max(h["recommendation_error"], 1e-6) for h in runs], name=method, marker_color=colors[method],
                         boxpoints="all", showlegend=False), row=1, col=2)
fig.update_yaxes(type="log", title_text="f - f*")
fig.update_xaxes(title_text="coût cumulé", row=1, col=1)
fig.update_layout(title=f"Optimisation de Hartmann 6D à budget égal ({BUDGET}), {len(SEEDS)} graines",
                  template="plotly_white", height=480)
fig.write_html(FIG_DIR / "optimization.html", include_plotlyjs="cdn")
fig

## 3. Vitesse de mfego avant / après la revue

Les temps ci-dessous viennent de `analysis/scripts/speedup_breakdown.py` (même machine) : 5 itérations EGO sur ce problème
(20 BF / 15 HF), code initial puis code corrigé avec chaque optimisation réactivée une à une (détail dans
`analysis/RAPPORT_ANALYSE.md`, §9). La dernière colonne de la table de la partie 2 donne le temps par itération des
différentes bibliothèques dans les mêmes conditions.

In [8]:
with open(ROOT / "analysis" / "results" / "speedup_breakdown.json", encoding="utf-8") as f:
    speed = json.load(f)
steps = pd.DataFrame(speed["ablation"])
steps["gain vs previous step"] = (steps["time_s"].shift(1) / steps["time_s"]).round(2)
steps["time_s"] = steps["time_s"].round(2)
display(steps)
fig = go.Figure(go.Bar(x=steps["time_s"], y=steps["step"], orientation="h", text=steps["time_s"],
                       marker={"color": ["#d62728", "#9467bd"] + ["#1f77b4"] * (len(steps) - 2)}))
fig.update_layout(title="mfego : temps de 5 itérations EGO (Hartmann 6D), effet cumulé de chaque optimisation",
                  xaxis_type="log", xaxis_title="s", yaxis={"autorange": "reversed"}, template="plotly_white",
                  height=420, margin={"l": 320})
fig

,step,time_s,gain vs previous step
0,Initial code (legacy/legacy_mfego_initial),46.41,NaN
1,"Corrected code, all optimizations OFF",24.85,1.87
2,+ N1 vectorized covariance matrix,6.14,4.05
3,+ N3 analytical gradient (log-space),4.29,1.43
4,"+ N4 Cholesky solves, batch prediction",4.45,0.96
5,+ N6 vectorized merit and DE,0.81,5.48
6,+ N8 warm start (= corrected code),0.69,1.17


## Conclusions

**Optimisation à budget égal** (coût 250, dont 120 de plan d'expériences initial, 5 graines) :

| Méthode | Erreur de la meilleure obs. HF (médiane) | Erreur de la recommandation, par graine | Médiane | Temps / itération |
|---|---|---|---|---|
| **mfego NN-MF-EGO** | **0,33** | 0,28 · 2,26 · 0,33 · 1,79 · 0,13 | **0,33** | 0,38 s |
| BoTorch qLogEI (SF) | 1,22 | 1,47 · 1,81 · 0,40 · 1,72 · 0,53 | 1,47 | 0,85 s |
| BoTorch MF-KG | 2,45 | 0,57 · 2,66 · 0,74 · 3,32 · 1,23 | 1,23 | 19 s |
| mfego SF-EGO | 1,74 | 2,76 · 1,76 · 2,02 · 1,55 · 1,85 | 1,85 | 0,15 s |

1. **mfego NN-MF-EGO obtient les meilleures médianes**, sur l'erreur de la meilleure observation HF comme sur celle de la
   recommandation. Il s'approche de l'optimum sur 3 graines sur 5 ; sur les 2 autres, il reste dans un bassin local de Hartmann,
   d'où une moyenne de 0,94-0,96. L'intérêt du multi-fidélité est net face aux méthodes mono-fidélité au même coût.
2. En mono-fidélité, **BoTorch qLogEI fait mieux que mfego SF-EGO** (1,22 contre 1,74 sur la meilleure observation) :
   l'optimisation de son acquisition (multi-départs à gradient) et ses priors sur les hyperparamètres sont plus efficaces que
   l'AEI de mfego optimisé par évolution différentielle.
3. **BoTorch MF-KG** choisit la basse fidélité dans environ 90 % des itérations ; il recommande un bon point sur 2 graines,
   mais coûte **19 s par itération**, environ 50 fois plus que mfego.
4. **Calibration** : avec peu de points, les modèles de BoTorch sont mieux calibrés que ceux de mfego (partie 1). Des priors
   faibles (MAP) sur les longueurs de corrélation et une borne basse relative du bruit sont la piste d'amélioration la plus
   directe pour mfego.
5. Ces conclusions valent pour un **petit budget** sur Hartmann 6D : 37 itérations en moyenne pour mfego, contre les 400 de
   Sacher et al. Les figures interactives sont dans `benchmarks/figures/`.